### Creating CollectionBuilder compatible metadata improvements

- changing object_id to objectid
- adding 'filename' column as /article_id/file (removing objects/ in front)

In [1]:
import os
import pandas as pd
import re


### Solving issue of duplicate rows

In [ ]:

INFILE  = "complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"

# --- helper functions ---
def slug(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s

def to_filename(p: str) -> str:
    p = (p or "").strip().replace("\\", "/")
    if not p:
        return ""
    if p.startswith("objects/"):
        return p[len("objects/"):]
    if p.startswith("/objects/"):
        return p[len("/objects/"):]
    return p

def img_index_from_filename(fn: str) -> str:
    fn = (fn or "").strip()
    m = re.search(r"(\d+)(?=\.[A-Za-z0-9]+$)", fn)
    return m.group(1) if m else ""

# --- load ---
df = pd.read_csv(INFILE, dtype=str).fillna("")
n0 = len(df)

# --- required columns ---
required = ["article_id", "image_display_template"]
for c in required:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

if "group_reprint_id" not in df.columns:
    raise ValueError("Expected column 'group_reprint_id' (you said you have it).")

# Create filename if needed
if "filename" not in df.columns:
    if "image_object_location" not in df.columns:
        raise ValueError("Need 'image_object_location' to derive filename.")
    df["filename"] = df["image_object_location"].apply(to_filename)

df["_article"]  = df["article_id"].apply(slug)
df["_context"]  = df["group_reprint_id"].apply(slug)
df.loc[df["_context"].str.strip() == "", "_context"] = df.loc[df["_context"].str.strip() == "", "_article"]

USE_REPRINT_TYPE_IN_CONTEXT = True
if USE_REPRINT_TYPE_IN_CONTEXT and "reprint_type" in df.columns:
    rt = df["reprint_type"].apply(slug)
    df["_context"] = df["_context"] + "__" + rt.where(rt.str.strip() != "", "na")

tmpl = df["display_template"].str.lower().str.strip()
is_parent = tmpl.eq("compound_object")
is_image  = tmpl.eq("image")

df.loc[is_parent, "objectid"] = df.loc[is_parent].apply(
    lambda r: f"{r['_article']}__{r['_context']}",
    axis=1
)

parent_lookup = {
    (r["_article"], r["_context"]): r["objectid"]
    for _, r in df[is_parent].iterrows()
}

def build_img_objectid(row):
    idx = ""
    for col in ["image_num", "img_num", "sequence", "order", "page"]:
        if col in df.columns and str(row[col]).strip():
            idx = str(row[col]).strip()
            break
    if not idx:
        idx = img_index_from_filename(row["filename"])
    if not idx:
        idx = slug(row["filename"])
    return f"{row['_article']}__{row['_context']}__img{idx}"

def parent_for_image(row):
    key = (row["_article"], row["_context"])
    return parent_lookup.get(key, "")

df.loc[is_image, "objectid"] = df.loc[is_image].apply(build_img_objectid, axis=1)
df.loc[is_image, "image_parent_id"] = df.loc[is_image].apply(parent_for_image, axis=1)

df = df.drop(columns=["_article", "_context"])

assert len(df) == n0, "Row count changed unexpectedly—abort!"

dupes = df[df["objectid"].duplicated(keep=False)]
print("Rows:", len(df))
print("Duplicate objectid rows after rebuild:", len(dupes))

if len(dupes) > 0:
    cols = [c for c in ["objectid","article_id","group_reprint_id","reprint_type","image_display_template","filename","image_parent_id"] if c in df.columns]
    print(dupes[cols].head(40).to_string(index=False))

df.to_csv(OUTFILE, index=False)
print("Wrote:", OUTFILE)


### Adding object_location column for CB 

In [ ]:

INFILE = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE  # overwrite

df = pd.read_csv(INFILE, dtype=str).fillna("")

def make_object_location(row):
    fn = (row.get("filename") or "").strip()
    if fn:
        return "/objects/" + fn
    loc = (row.get("image_object_location") or "").strip()
    if loc:
        loc = loc.replace("\\", "/")
        return loc if loc.startswith("/") else "/" + loc
    return ""

df["object_location"] = df.apply(make_object_location, axis=1)

if "format" not in df.columns:
    df["format"] = ""
mask_img = df.get("image_display_template", "").str.lower().eq("image")
df.loc[mask_img, "format"] = "image"

df.to_csv(OUTFILE, index=False)
print("Added object_location (+ format=image for image rows).")


### Adding diplay_template column

In [ ]:

INFILE = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")

df = df.rename(columns={"image_display_template": "display_template"})

df.to_csv(OUTFILE, index=False)
print("Added display_template column.")


## Adding image format column

In [ ]:
import pandas as pd

INFILE = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")
if "format" not in df.columns:
    df["format"] = ""

tmpl = df["display_template"].str.lower().str.strip()
df.loc[tmpl.eq("image"), "format"] = "image"
df.loc[tmpl.eq("compound_object"), "format"] = "multiple"

df.to_csv(OUTFILE, index=False)
print("Set format for image + compound rows.")


In [ ]:
### changing object_id column to object_image_tag



In [ ]:

INFILE = "../_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")

if "object_id" in df.columns:
    df = df.rename(columns={"object_id": "article_image_tag"})
    print("Renamed column: object_id -> article_image_tag")
else:
    print("Column 'object_id' not found — no rename performed.")

print("Columns:", df.columns.tolist())
df.to_csv(OUTFILE, index=False)
print("Wrote:", OUTFILE)
